# Smart speaker demo video: McNemar test

**McNemar Test** · a worked example from the StatNex teaching guide

**The story.** 100 shoppers were asked whether they would buy a new smart speaker, then watched a one-minute demo video and were asked again. The same people answered twice.

**The question.** Did the demo video change people's minds?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/mcnemar_test.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Before the video,After: would buy,After: would not buy,People
Before: would buy,40.0,3.0,43.0
Before: would not buy,22.0,35.0,57.0
Total,62.0,38.0,100.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.contingency_tables import SquareTable, mcnemar

pd.set_option("display.width", 120)

## The data

In [ ]:
# A 2 x 2 table of the same subjects judged twice -- or k x k, with the same k
# categories as rows and columns: rows are one measurement, columns the other.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["After: would buy", "After: would not buy"]
table = data.set_index("Before the video")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
from totals import is_total

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()

## Observed frequency

In [ ]:
observed = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
observed = pd.concat([observed, observed.sum().to_frame("Total").T])
print(observed)

## McNemar test

In [ ]:
# Only the two cells where the measurements disagree (b and c) enter the test. With
# the continuity correction, chi-square = (|b - c| - 1)^2 / (b + c) on 1 degree of
# freedom; correction=False would give (b - c)^2 / (b + c).
# How to read it: if the two measurements had the same rate of the first category, a
# case that changed would be as likely to change one way as the other, so b and c would
# be about equal. A small P says the changes lean one way. The correction makes the
# test a little cautious; with b + c below about 25, the exact binomial version
# (b out of b + c, probability 0.5) is the safer choice.
counts = table.to_numpy(float)
if counts.shape == (2, 2):
    result = mcnemar(counts, exact=False, correction=True)
    test = pd.DataFrame([[result.statistic, 1, result.pvalue]], columns=["chi-square", "df", "P"])
else:
    # k x k: Bowker's test that the table is symmetric, summing (n_ij - n_ji)^2 /
    # (n_ij + n_ji) over the pairs either side of the diagonal, on one df per pair that has
    # any cases: an empty pair carries no information. (statsmodels' SquareTable.symmetry
    # keeps k(k-1)/2 df whatever, which makes the test conservative.)
    upper = np.triu_indices(counts.shape[0], 1)
    sums = (counts + counts.T)[upper]
    gaps = (counts - counts.T)[upper]
    statistic = (gaps[sums > 0] ** 2 / sums[sums > 0]).sum()
    df = int((sums > 0).sum())
    test = pd.DataFrame([[statistic, df, stats.chi2.sf(statistic, df)]],
                        columns=["chi-square", "df", "P"])
print()
print(test.to_string(index=False))

if counts.shape != (2, 2):
    # === Marginal homogeneity (Stuart-Maxwell) ===
    # Symmetry implies equal margins but not the reverse, so this asks the weaker
    # question: do the two measurements put the same share of cases in each category?
    result = SquareTable(counts, shift_zeros=False).homogeneity(method="stuart_maxwell")
    homogeneity = pd.DataFrame([[result.statistic, result.df, result.pvalue]],
                               columns=["chi-square", "df", "P"])
    print(homogeneity.to_string(index=False))

## What the result says

The same 100 shoppers answered twice, so the McNemar test looks only at the people who changed their answer. After the video, 22 shoppers switched from would not buy to would buy, and only 3 switched the other way. The others did not change.

If the video had no effect, switches in the two directions would be about equal. Chi-square is 12.96 with 1 degree of freedom and P = 0.0003. A P value is the chance of a gap at least this large with no real effect, so chance is a poor explanation.

So the video moved the answers: 62 shoppers say they would buy after it, up from 43. Whether the video sells more speakers is another question, because saying is not buying.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.